# Agentic RAG — Exploration Notebook
**Meridian Capital Group HR Policy Q&A**

This notebook walks through the full agentic RAG system — from document ingestion through the LangGraph state graph, query routing, tool calls, memory, and evaluation. It builds on the baseline `rag-pipeline` notebook and focuses on what changes when you move from passive retrieval to an active reasoning agent.

Sections:
0. Setup
1. Document Ingestion & Chunking (LlamaIndex → LangChain bridge)
2. Embeddings & Pinecone Vector Store
3. Query Routing — RouteDecision inspection
4. Agent State Graph — nodes and edges walkthrough
5. Tool Calls — retriever, calculator, date lookup
6. Conversational Memory — sliding window + summarization
7. End-to-End Agent Runs — traced reasoning steps
8. RAG vs Agentic RAG — side-by-side comparison
9. Evaluation — agentic metrics
10. Next Steps

## 0. Setup

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

# Verify keys are present
for key in ['OPENAI_API_KEY', 'PINECONE_API_KEY', 'PINECONE_INDEX_NAME']:
    status = '✓' if os.getenv(key) else '✗ MISSING'
    print(f'{key}: {status}')

## 1. Document Ingestion & Chunking

The agentic-rag pipeline uses a bridge pattern: `loader.py` uses LlamaIndex `SimpleDirectoryReader` (best-in-class PDF handling), then `chunker.py` converts the output to LangChain `Document` objects so the rest of the pipeline stays in the LangChain ecosystem.

Key difference from rag-pipeline: chunk size is 384 chars / 40 overlap (character-based via `RecursiveCharacterTextSplitter`) vs 512 tokens / 50 overlap (token-based via `SentenceSplitter`).

In [ ]:
from src.ingestion.loader import load_documents
from src.ingestion.chunker import chunk_documents

# Load with LlamaIndex
llamaindex_docs = load_documents('../data/raw')
print(f'LlamaIndex documents loaded : {len(llamaindex_docs)}')
print(f'Type                        : {type(llamaindex_docs[0])}')

# Convert to LangChain Documents
lc_chunks = chunk_documents(llamaindex_docs)
print(f'\nLangChain chunks produced   : {len(lc_chunks)}')
print(f'Type                        : {type(lc_chunks[0])}')
print(f'\nSample chunk (first 300 chars):\n{lc_chunks[0].page_content[:300]}')

In [ ]:
# Compare chunk size distributions
lengths = [len(c.page_content) for c in lc_chunks]
print(f'Chunk count : {len(lengths)}')
print(f'Min chars   : {min(lengths)}')
print(f'Max chars   : {max(lengths)}')
print(f'Avg chars   : {sum(lengths)/len(lengths):.0f}')

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3))
    plt.hist(lengths, bins=20, color='#10b981', edgecolor='white')
    plt.title('Chunk length distribution (chars)')
    plt.xlabel('Characters')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

## 2. Embeddings & Pinecone Vector Store

Embeddings use `OpenAIEmbeddings` (LangChain) with `text-embedding-3-small` (1536 dims). The vector store is Pinecone serverless — `store.py` handles both cold start (upsert from documents) and warm start (connect to existing index) via `from_documents()` and `from_existing_index()`.

In [ ]:
from src.embedding.embedder import get_embedder, embed_texts, embed_query

embedder = get_embedder()
print(f'Model : {embedder.model}')

# Embed a sample query
q = 'How many PTO days does a new employee receive?'
vec = embed_query(embedder, q)
print(f'Query : {q}')
print(f'Vector dims  : {len(vec)}')
print(f'First 5 vals : {[round(v, 4) for v in vec[:5]]}')

In [ ]:
from src.vectorstore.store import get_vector_store

# Warm start — connect to existing Pinecone index
# Set FORCE_REBUILD=true in .env to re-upsert all chunks
vectorstore = get_vector_store(chunks=lc_chunks, embedder=embedder)
print(f'Vectorstore type : {type(vectorstore)}')
print(f'Index name       : {os.getenv("PINECONE_INDEX_NAME")}')

In [ ]:
from src.retrieval.retriever import get_retriever, retrieve_with_scores

retriever = get_retriever(vectorstore)

# Test retrieval with scores
results = retrieve_with_scores(vectorstore, q, k=4)
print(f'Query: {q}\n')
for i, (doc, score) in enumerate(results):
    print(f'[{i+1}] score={score:.4f} | {doc.page_content[:120]}...')

## 3. Query Routing

`router.py` uses `with_structured_output()` to force the LLM to return a typed `RouteDecision` object — no string parsing, no regex. Each decision includes a route type, confidence score (0–1), and reasoning string.

Route types: `hr_retrieval`, `calculation`, `date_lookup`, `multi_step`, `escalation`, `out_of_scope`.

In [ ]:
from src.agents.router import QueryRouter

router = QueryRouter()

test_queries = [
    'How many PTO days does a new employee get?',          # hr_retrieval
    'If I take 5 sick days and 3 PTO days, how many do I have left?',  # calculation
    'How long until my 401k is fully vested if I started in March 2023?',  # date_lookup + multi_step
    'What is the weather like today?',                    # out_of_scope
    'I want to report my manager for harassment',          # escalation
]

for q in test_queries:
    decision = router.classify(q)
    print(f'Query      : {q}')
    print(f'Route      : {decision.route}')
    print(f'Confidence : {decision.confidence:.2f}')
    print(f'Reasoning  : {decision.reasoning}')
    print()

In [ ]:
# Inspect low-confidence routing — ambiguous queries
ambiguous = [
    'Can you help me understand my benefits?',
    'What happens to my RSUs?',
]

for q in ambiguous:
    decision = router.classify(q)
    flag = '⚠ LOW CONFIDENCE' if decision.confidence < 0.7 else ''
    print(f'Query      : {q}')
    print(f'Route      : {decision.route}  {flag}')
    print(f'Confidence : {decision.confidence:.2f}')
    print()

## 4. Agent State Graph

`agent.py` builds a LangGraph `StateGraph` with typed `AgentState`. Each node is a pure function — dependencies (retriever, router, generator, memory) are injected via `functools.partial` at `build_graph()` time, keeping nodes testable in isolation.

**Nodes:** `route_query` → `retrieve` / `calculate` / `date_lookup` / `decompose_multi_step` / `handle_escalation` / `handle_out_of_scope` → `check_retrieval` → `generate` → `update_memory`

**Conditional edges:** `route_to_node` (after routing), `after_retrieval` (score check + retry loop bounded by `MAX_AGENT_ITERATIONS`), `after_tool_or_escalation`

In [ ]:
from src.agents.agent import build_graph
from src.memory.memory import ConversationMemory

memory = ConversationMemory()
graph = build_graph(
    vectorstore=vectorstore,
    embedder=embedder,
    memory=memory
)

print('Graph compiled successfully')
print(f'Nodes: {list(graph.nodes.keys())}')

In [ ]:
# Print ASCII graph structure
from src.pipeline import AgenticRAGPipeline

pipeline = AgenticRAGPipeline()
print(pipeline.get_graph_diagram())

## 5. Tool Calls

`tools.py` registers four tools with the `@tool` decorator via a factory pattern in `get_tools()`. All tools return strings as required by LangGraph's `ToolNode`.

- `hr_policy_retriever` — wraps the Pinecone retriever
- `calculator` — evaluates arithmetic expressions from the query
- `date_calculator` — computes date differences and future dates
- `escalation_router` — flags sensitive queries for human review

In [ ]:
from src.agents.tools import get_tools

tools = get_tools(vectorstore=vectorstore, embedder=embedder)
print(f'Tools registered: {[t.name for t in tools]}\n')

# Call each tool directly
hr_tool = next(t for t in tools if t.name == 'hr_policy_retriever')
calc_tool = next(t for t in tools if t.name == 'calculator')
date_tool = next(t for t in tools if t.name == 'date_calculator')

print('--- hr_policy_retriever ---')
print(hr_tool.invoke({'query': 'parental leave policy'})[:400])

print('\n--- calculator ---')
print(calc_tool.invoke({'expression': '15 + 5 - 3'}))

print('\n--- date_calculator ---')
print(date_tool.invoke({'query': 'How long until March 2023 is 4 years ago?'}))

## 6. Conversational Memory

`ConversationMemory` implements a sliding window with automatic summarization. When the turn count exceeds `MEMORY_SUMMARY_THRESHOLD` (default 10), older turns are summarized into a single context string and dropped from the window. Memory context is injected into `AgentState` at query time — not appended to the prompt chain.

Each `Turn` stores the query, response, route taken, and any tool calls made.

In [ ]:
from src.memory.memory import ConversationMemory

mem = ConversationMemory(max_turns=5, summary_threshold=4)

# Simulate a conversation
exchanges = [
    ('How many PTO days do I get?', 'New employees receive 15 days per year.', 'hr_retrieval'),
    ('What about sick days?', 'You receive 7 sick days per year.', 'hr_retrieval'),
    ('When can I use them?', 'PTO is available after a 90-day waiting period.', 'hr_retrieval'),
    ('Can I carry them over?', 'You can carry over up to 10 days per year.', 'hr_retrieval'),
]

for query, response, route in exchanges:
    mem.add_turn(query=query, response=response, route=route)

print(f'Turns stored : {len(mem.turns)}')
print(f'\nFormatted context injected into AgentState:')
print(mem.format_context())

In [ ]:
# Trigger summarization by exceeding threshold
mem.add_turn(
    query='Do PTO days accrue monthly?',
    response='Yes, PTO accrues monthly based on your tenure tier.',
    route='hr_retrieval'
)

print(f'Turns after summarization : {len(mem.turns)}')
print(f'Summary generated         : {bool(mem.summary)}')
if mem.summary:
    print(f'\nSummary:\n{mem.summary}')

## 7. End-to-End Agent Runs

We run queries through the full pipeline and inspect the `AgentState` at each step — route taken, tool calls made, retrieval scores, iterations, and final response.

In [ ]:
from src.pipeline import AgenticRAGPipeline

pipeline = AgenticRAGPipeline()

test_queries = [
    'How many PTO days does a new employee receive?',
    'If I have 15 PTO days and use 7, how many remain?',
    'Does equity vesting continue during parental leave?',
    'What is the meaning of life?',
]

for q in test_queries:
    result = pipeline.query(q)
    pipeline.print_response(result)
    print('-' * 60)

In [ ]:
# Inspect the full AgentState for a complex query
multi_step_q = 'I started in January 2023. When will my 401k be fully vested and how much will Meridian have contributed if I contribute the max?'

result = pipeline.query(multi_step_q)

print(f'Query          : {multi_step_q}')
print(f'Route          : {result["route"]}')
print(f'Confidence     : {result.get("route_confidence", "n/a")}')
print(f'Sub-queries    : {result.get("sub_queries", [])}')
print(f'Tool calls     : {result.get("tool_calls", [])}')
print(f'Iterations     : {result.get("iterations", 1)}')
print(f'\nResponse:\n{result["response"]}')

## 8. RAG vs Agentic RAG

We compare the baseline RAG pipeline response against the agentic response on queries that require reasoning beyond simple retrieval — calculations, multi-step logic, and ambiguous or sensitive queries.

In [ ]:
from src.generation.generator import get_llm, generate
from src.retrieval.retriever import retrieve_with_scores, retrieve_with_metadata

llm = get_llm()

comparison_queries = [
    'If I joined in June 2022, when does my 401k fully vest?',
    'I want to report a colleague for inappropriate behaviour.',
]

for q in comparison_queries:
    # Baseline RAG — retrieve and generate
    context_docs = retrieve_with_metadata(vectorstore, q, k=3)
    context_str = '\n\n'.join([d.page_content for d in context_docs])
    baseline_ans = generate(llm, q, context_str)

    # Agentic RAG
    agentic_result = pipeline.query(q)

    print(f'Query: {q}')
    print(f'\n[Baseline RAG]\n{baseline_ans}')
    print(f'\n[Agentic RAG — route: {agentic_result["route"]}]\n{agentic_result["response"]}')
    print('=' * 70)

## 9. Evaluation

We run a subset of the ground truth Q&A pairs and measure agentic-specific metrics alongside the standard RAG metrics:

- **Route accuracy** — did the router select the correct route for each query type?
- **Tool precision** — when a tool was called, was it the right one?
- **Avg iterations** — how many retrieval attempts were needed per query?
- **Escalation rate** — what proportion of sensitive queries were correctly escalated?
- **Faithfulness / Relevance / Correctness** — scored 1–5 by LLM-as-judge against ground truth

In [ ]:
import json
from collections import defaultdict
from statistics import mean

with open('../evaluation/qa_pairs.json') as f:
    eval_data = json.load(f)

pairs = eval_data['pairs']
print(f'Total Q&A pairs : {len(pairs)}')
print(f'Difficulty split:')
for diff in ['easy', 'medium', 'hard']:
    print(f'  {diff}: {len([p for p in pairs if p["difficulty"] == diff])}')

In [ ]:
# Run on a small sample — 3 easy, 2 medium, 1 hard
sample = (
    [p for p in pairs if p['difficulty'] == 'easy'][:3] +
    [p for p in pairs if p['difficulty'] == 'medium'][:2] +
    [p for p in pairs if p['difficulty'] == 'hard'][:1]
)

results = []
for pair in sample:
    result = pipeline.query(pair['question'])
    results.append({
        'question'   : pair['question'],
        'expected'   : pair['answer'],
        'got'        : result['response'],
        'route'      : result['route'],
        'iterations' : result.get('iterations', 1),
        'difficulty' : pair['difficulty'],
    })
    print(f'[{pair["difficulty"]:6}] {pair["question"][:70]}')
    print(f'         route={result["route"]} | iters={result.get("iterations", 1)}')
    print()

In [ ]:
# Aggregate by difficulty and route
by_difficulty = defaultdict(list)
by_route = defaultdict(list)

for r in results:
    by_difficulty[r['difficulty']].append(r['iterations'])
    by_route[r['route']].append(r['iterations'])

print('Avg iterations by difficulty:')
for diff, iters in by_difficulty.items():
    print(f'  {diff}: {mean(iters):.1f}')

print('\nAvg iterations by route:')
for route, iters in by_route.items():
    print(f'  {route}: {mean(iters):.1f}')

print(f'\nRun full eval suite:')
print('  python evaluation/eval.py --qa evaluation/qa_pairs.json --output evaluation/results.json')

## 10. Next Steps

**Run the full evaluation suite:**
```bash
python evaluation/eval.py --qa evaluation/qa_pairs.json --output evaluation/results.json
```

**Launch the Streamlit UI:**
```bash
python ui/app.py
```

**Planned additions:**
- RAGAS evaluation framework — standardised metrics including agentic-specific measures
- NeMo Guardrails — at graph entry (before `route_query`) and exit (before response)

**Related:** See `../rag-pipeline/notebooks/exploration.ipynb` for the baseline RAG walkthrough this builds on.